In [1]:
import numpy as np 
import pandas as pd 

In [2]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings('ignore')

In [3]:
# Verileri Kaggle yollarından çekiyoruz
try:
    train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
    test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')
    print(f"Veri yüklendi. Train: {train.shape}, Test: {test.shape}")
except:
    # Eğer lokalde çalışıyorsan
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test_x.csv')
    print("Yerel dosyalar yüklendi.")

Veri yüklendi. Train: (56000, 24), Test: (24000, 23)


In [4]:
def advanced_feature_engineering(df, is_train=True):
    df = df.copy()
    
    # Kritik Eksikleri Doldurma
    df['vucut_kitle_indeksi'] = df['vucut_kitle_indeksi'].fillna(train['vucut_kitle_indeksi'].median())
    df['yas'] = df['yas'].fillna(train['yas'].median())
    
    # Gruplama (Binning) 
    df['yas_grubu'] = pd.cut(df['yas'], bins=[0, 25, 35, 45, 55, 100], labels=[0, 1, 2, 3, 4]).astype(int)
    df['vki_grubu'] = pd.cut(df['vucut_kitle_indeksi'], bins=[0, 18.5, 25, 30, 100], labels=[0, 1, 2, 3]).astype(int)

    
    df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['uyku_bozucu_skor'] = df['uyku_oncesi_kafein_mg'].fillna(0) + df['uyku_oncesi_ekran_suresi_dk'].fillna(0)
    df['aktivite_denge'] = df['gunluk_adim_sayisi'] / (df['gunluk_calisma_saati'] + 1)
    df['stres_uyanma'] = df['stres_skoru'].fillna(df['stres_skoru'].median()) * df['gecelik_uyanma_sayisi']
    df['uyku_duzensizligi'] = df['hafta_sonu_uyku_farki_saat'].abs()
    
    # 3. Kalan Eksik Değerleri Doldurma
    kat_sutunlar = ['meslek', 'kronotip', 'ruh_sagligi_durumu']
    for col in kat_sutunlar:
        df[col] = df[col].fillna(train[col].mode()[0])
        
    sayisal_eksikler = df.select_dtypes(include=[np.number]).columns
    for col in sayisal_eksikler:
        df[col] = df[col].fillna(train[col].median() if col in train.columns else df[col].median())
        
    return df

train_fe = advanced_feature_engineering(train)
test_fe = advanced_feature_engineering(test)

# Kategorik Dönüştürme
le = LabelEncoder()
kategorik_kolonlar = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']

for col in kategorik_kolonlar:
    combined = pd.concat([train_fe[col], test_fe[col]], axis=0).astype(str)
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))

# X ve y Ayırma
X = train_fe.drop(columns=['id', 'bilissel_performans_skoru'])
y = train_fe['bilissel_performans_skoru']
X_test = test_fe.drop(columns=['id'])

In [5]:
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Tahminleri toplamak için diziler
lgb_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))
oof_ensemble = np.zeros(len(X)) # Skor tahmini için

print("Model Eğitimi ve K-Fold Doğrulama Başlıyor...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # 1. LightGBM (Hassas Ayarlı)
    lgbm = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.01, num_leaves=63, feature_fraction=0.8, random_state=42, verbose=-1)
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
    
    # 2. XGBoost
    xgbr = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.01, max_depth=7, subsample=0.8, colsample_bytree=0.8, random_state=42)
    xgbr.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    
    # 3. CatBoost
    catb = CatBoostRegressor(n_estimators=2000, learning_rate=0.01, depth=7, random_state=42, verbose=0, early_stopping_rounds=100)
    catb.fit(X_tr, y_tr)
    
    # Fold tahminlerini alma
    fold_lgb = lgbm.predict(X_val)
    fold_xgb = xgbr.predict(X_val)
    fold_cat = catb.predict(X_val)
    
    # OOF (Skor tahmini) için birleştirme
    oof_ensemble[val_idx] = (fold_lgb * 0.40) + (fold_cat * 0.45) + (fold_xgb * 0.15)
    
    # Test seti için tahminleri biriktir
    lgb_preds += lgbm.predict(X_test) / N_SPLITS
    xgb_preds += xgbr.predict(X_test) / N_SPLITS
    cat_preds += catb.predict(X_test) / N_SPLITS
    
    print(f"Fold {fold+1} tamamlandı.")

# TAHMİNİ SKOR HESAPLAMA
total_rmse = np.sqrt(mean_squared_error(y, oof_ensemble))
print(f"\n--- YENİ TAHMİNİ YARIŞMA SKORUN (RMSE): {total_rmse:.4f} ---")

Model Eğitimi ve K-Fold Doğrulama Başlıyor...
Fold 1 tamamlandı.
Fold 2 tamamlandı.
Fold 3 tamamlandı.
Fold 4 tamamlandı.
Fold 5 tamamlandı.

--- YENİ TAHMİNİ YARIŞMA SKORUN (RMSE): 1.2229 ---


In [6]:
final_predictions = (lgb_preds * 0.40) + (cat_preds * 0.45) + (xgb_preds * 0.15)

submission = pd.DataFrame({
    'id': test['id'],
    'bilissel_performans_skoru': final_predictions
})

submission.to_csv('final_pro_submission.csv', index=False)
print("Dosya başarıyla kaydedildi! Kaggle'a yüklemeye hazır.")

Dosya başarıyla kaydedildi! Kaggle'a yüklemeye hazır.
